## Simulating traffic passing multiple locks
In this notebook, we simulate two locks on a network which vessels from opposing direction shave to pass. Vessels are locked together if they can fit inside the lock, and arrive within the clustering time window.

#### 0. Import libraries

In [1]:
# package(s) used for creating and geo-locating the graph
import networkx as nx
import pyproj
from shapely.geometry import Point, LineString
from shapely.ops import transform

# package(s) related to the simulation (creating the vessel, running the simulation)
import datetime
import simpy
import opentnsim
from opentnsim.core.logutils import logbook2eventtable
from opentnsim.core.visualizations import generate_vessel_gantt_chart
from scipy.stats import norm, uniform, expon

# import of modules important for locking
from opentnsim.lock import lock as lock_module
from opentnsim.graph import mixins as graph_module

# package(s) needed for inspecting the output
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("This notebook is executed with OpenTNSim version {}".format(opentnsim.__version__))

This notebook is executed with OpenTNSim version 1.3.4


#### 0. Create environment

In [2]:
# start simpy environment
simulation_start = datetime.datetime(2025, 1, 1, 0, 0, 0)
env = simpy.Environment(initial_time=simulation_start.timestamp())
env.epoch = simulation_start

#### 1. Create graph

In [3]:
# define reference systems
wgs84eqd = pyproj.CRS('4087')
wgs84rad = pyproj.CRS('4326')

# define transformer functions
wgs84eqd_to_wgs84rad = pyproj.transformer.Transformer.from_crs(wgs84eqd,wgs84rad,always_xy=True).transform #equidistant wgs84 to radial wgs84
wgs84rad_to_wgs84eqd = pyproj.transformer.Transformer.from_crs(wgs84rad,wgs84eqd,always_xy=True).transform #radial wgs84 to equidistant wgs84

# create a directed graph
graph = nx.DiGraph()

# add nodes
graph.add_node('-2',geometry=transform(wgs84eqd_to_wgs84rad,Point(-350600,0)))
graph.add_node('-1',geometry=transform(wgs84eqd_to_wgs84rad,Point( -15000,0)))
graph.add_node('0',geometry=transform(wgs84eqd_to_wgs84rad,Point(-5000,0)))
graph.add_node('1',geometry=transform(wgs84eqd_to_wgs84rad,Point(5000,0)))
graph.add_node('+1',geometry=transform(wgs84eqd_to_wgs84rad,Point(15000,0)))
graph.add_node('+2',geometry=transform(wgs84eqd_to_wgs84rad,Point(350600,0)))

# add edges
graph.add_edge('-2','-1', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(-350600, 0),Point(-15000, 0)])), weight=1, length_m=350600)
graph.add_edge('-1','-2', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(-15000, 0),Point(-350600, 0)])), weight=1, length_m=350600)

graph.add_edge('-1','0', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(-15000, 0),Point(-5000, 0)])), weight=1, length_m=10000)
graph.add_edge('0','-1', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(-5000, 0),Point(-15000, 0)])), weight=1, length_m=10000)

graph.add_edge('0','1', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(-5000, 0),Point(5000, 0)])), weight=1, length_m=10000)
graph.add_edge('1','0', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(5000, 0),Point(-5000, 0)])), weight=1, length_m=10000)

graph.add_edge('1','+1', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(5000, 0),Point(15000, 0)])), weight=1, length_m=10000)
graph.add_edge('+1','1', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(15000, 0),Point(5000, 0)])), weight=1, length_m=10000)

graph.add_edge('+2','+1', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(15000, 0),Point(350600, 0)])), weight=1, length_m=350600)
graph.add_edge('+1','+2', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(350600, 0),Point(15000, 0)])), weight=1, length_m=350600)

# add graph to environment
env.graph = graph

In [4]:
graph_module.plot_graph(graph)

#### 1+ Adding infrastructure

In [5]:
lock_1 = lock_module.IsLockComplex(
    env=env,
    name='Lock_1',
    node_open='-1',
    node_A = '-1',
    node_B = '0',
    distance_lock_doors_A_to_waiting_area_A = 4800,
    distance_lock_doors_B_to_waiting_area_B = 4800,
    distance_from_start_node_to_lock_doors_A = 4800,
    distance_from_end_node_to_lock_doors_B = 4800,
    lock_length = 400,
    lock_width = 50,
    lock_depth = 15,
    levelling_time = 300,
    sailing_distance_to_crossing_point = 1800,
    doors_opening_time= 300,
    doors_closing_time= 300,
    speed_reduction_factor_lock_chamber=0.5,
    sailing_in_time_gap_through_doors = 300,
    sailing_in_speed_sea = 1.5,
    sailing_in_speed_canal = 1.5,
    sailing_out_time_gap_through_doors = 120,
    sailing_time_before_opening_lock_doors = 600,
    sailing_time_before_closing_lock_doors = 120,
    registration_nodes = ['-2','1'],
    predictive=False
)

lock_2 = lock_module.IsLockComplex(
    env=env,
    name='Lock_2',
    node_open='1',
    node_A = '1',
    node_B = '+1',
    distance_lock_doors_A_to_waiting_area_A = 4800,
    distance_lock_doors_B_to_waiting_area_B = 4800,
    distance_from_start_node_to_lock_doors_A = 4800,
    distance_from_end_node_to_lock_doors_B = 4800,
    lock_length = 400,
    lock_width = 50,
    lock_depth = 15,
    levelling_time = 300,
    sailing_distance_to_crossing_point = 1800,
    doors_opening_time= 300,
    doors_closing_time= 300,
    speed_reduction_factor_lock_chamber=0.5,
    sailing_in_time_gap_through_doors = 300,
    sailing_in_speed_sea = 1.5,
    sailing_in_speed_canal = 1.5,
    sailing_out_time_gap_through_doors = 120,
    sailing_time_before_opening_lock_doors = 600,
    sailing_time_before_closing_lock_doors = 120,
    registration_nodes = ['0','+2'],
    predictive=False
)

#### 2. Create agents

In [6]:
# make your preferred Vessel class out of available mix-ins.
Vessel = type(
    "Vessel", 
    (
        lock_module.PassesLockComplex,             # allows to interact with a lock
        opentnsim.core.Identifiable,               # allows to give the object a name and a random ID,
        opentnsim.core.Movable,                    # allows the object to move, with a fixed speed, while logging this activity
        opentnsim.core.VesselProperties,           # allows vessel to have dimensions, namely a length (L), width (B), and draught (T)
        opentnsim.core.ExtraMetadata,              # allow additional information, such as an arrival time (required for passing a lock)
        graph_module.HasMultiDiGraph,           # allow to operate on a graph that can include parallel edges from and to the same nodes
        opentnsim.output.HasOutput,                # allow additional output to be stored
    ), 
    {}
)

In [7]:
def mission(env, vessel):
    """
    Method that defines the mission of the vessel.
    
    In this case: 
        keep moving along the path until its end point is reached
    """
    while True:
        yield from vessel.move()
        
        if vessel.geometry == nx.get_node_attributes(env.graph, "geometry")[vessel.route[-1]]:
            break

In [8]:
# create vessels from dict 
data_vessel_1 = {
    "env": env,                                          # needed for simpy simulation
    "name": "Vessel 1",                                  # required by Identifiable
    "geometry": env.graph.nodes['-2']['geometry'],       # required by Locatable
    "route": nx.dijkstra_path(env.graph, "-2", "+2"),    # required by Routeable
    "v": 4,                                              # required by Movable, 4 m/s to check if the distance is covered in the expected time
    "L": 100,                                            # required by VesselProperties, interacts with the lock capacity
    "B": 20,                                             # required by VesselProperties
    "T": 10,                                             # required by VesselProperties
    "type": 'tanker',                                    # required by VesselProperties
    "arrival_time": pd.Timestamp('2025-01-01 00:00:00')  # required by PassesLockComplex
}  
vessel_1 = Vessel(**data_vessel_1)
vessel_1.name = 'Vessel 1'

data_vessel_2 = {
    "env": env,                                          # needed for simpy simulation
    "name": "Vessel 2",                                  # required by Identifiable
    "geometry": env.graph.nodes['+2']['geometry'],       # required by Locatable
    "route": nx.dijkstra_path(env.graph, "+2", "-2"),    # required by Routeable
    "v": 4,                                              # required by Movable, 4 m/s to check if the distance is covered in the expected time
    "L": 100,                                            # required by VesselProperties, interacts with the lock capacity
    "B": 20,                                             # required by VesselProperties
    "T": 10,                                             # required by VesselProperties
    "type": 'tanker',                                    # required by VesselProperties
    "arrival_time": pd.Timestamp('2025-01-01 00:05:00')  # required by PassesLockComplex
}  
vessel_2 = Vessel(**data_vessel_2)
vessel_2.name = 'Vessel 2'

#start the simulation
env.vessels = [vessel_1,vessel_2]
env.process(mission(env, vessel_1));
env.process(mission(env, vessel_2));

#### 3. Run simulation

In [9]:
env.run()

#### 4. Inspect output

In [10]:
# load the logbook data into a dataframe
lock_df = pd.DataFrame.from_dict(lock_1.lock_chamber.logbook)

print("'{}' logbook data:".format(lock_1.name))  
print('')

display(lock_df)

'Lock_1' logbook data:



,Message,Timestamp,Value,Geometry
0,Lock doors closing start,2025-01-02 00:46:30.172786,{},-1
1,Lock doors closing stop,2025-01-02 00:51:30.172786,{},-1
2,Lock chamber converting start,2025-01-02 00:51:30.172786,{},-1
3,Lock chamber converting stop,2025-01-02 00:56:30.172786,{},0
4,Lock doors opening start,2025-01-02 00:56:30.172786,{},0
5,Lock doors opening stop,2025-01-02 01:01:30.172786,{},0
6,Lock doors closing start,2025-01-02 02:34:39.000000,{},0
7,Lock doors closing stop,2025-01-02 02:39:39.000000,{},0
8,Lock chamber converting start,2025-01-02 02:39:39.000000,{},0
9,Lock chamber converting stop,2025-01-02 02:44:39.000000,{},-1


In [11]:
# load the logbook data into a dataframe
lock_df = pd.DataFrame.from_dict(lock_2.lock_chamber.logbook)

print("'{}' logbook data:".format(lock_2.name))  
print('')

display(lock_df)

'Lock_2' logbook data:



,Message,Timestamp,Value,Geometry
0,Lock doors closing start,2025-01-01 00:05:00.000000,{},1
1,Lock doors closing stop,2025-01-01 00:10:00.000000,{},1
2,Lock chamber converting start,2025-01-01 00:10:00.000000,{},1
3,Lock chamber converting stop,2025-01-01 00:15:00.000000,{},+1
4,Lock doors opening start,2025-01-01 00:15:00.000000,{},+1
5,Lock doors opening stop,2025-01-01 00:20:00.000000,{},+1
6,Lock doors closing start,2025-01-02 00:51:30.172786,{},+1
7,Lock doors closing stop,2025-01-02 00:56:30.172786,{},+1
8,Lock chamber converting start,2025-01-02 00:56:30.172786,{},+1
9,Lock chamber converting stop,2025-01-02 01:01:30.172786,{},1


#### Gantt chart of event table

In [12]:
df_eventtable = opentnsim.core.logutils.logbook2eventtable([*env.vessels, lock_1.lock_chamber, lock_2.lock_chamber])
fig = generate_vessel_gantt_chart(df_eventtable)

#### Time-distance diagram of vessels passing the lock and planning info

In [13]:
def cm_to_pixels(cm):
    return cm * 37.8 # Set figure height to 10 cmfig.update_layout(height=cm_to_pixels(10))

# We can plot the time-distance diagram
fig = lock_1.create_time_distance_plot(vessels = env.vessels, 
                                       xlimmin = -6050, 
                                       xlimmax = 6050,
                                       ylimmin = pd.Timestamp('2025-01-01 22:00:00'),
                                       ylimmax = pd.Timestamp('2025-01-02 09:00:00'),
                                       method='Plotly')

fig.update_layout(height=cm_to_pixels(20))

In [14]:
def cm_to_pixels(cm):
    return cm * 37.8 # Set figure height to 10 cmfig.update_layout(height=cm_to_pixels(10))

# We can plot the time-distance diagram
fig = lock_2.create_time_distance_plot(vessels = env.vessels, 
                                       xlimmin = -6050, 
                                       xlimmax = 6050,
                                       ylimmin = pd.Timestamp('2025-01-01 22:00:00'),
                                       ylimmax = pd.Timestamp('2025-01-02 09:00:00'),
                                       method='Plotly')

fig.update_layout(height=cm_to_pixels(20))

#### Vessel delays: individual delays and overall average

In [15]:
delays = []
for vessel in env.vessels:
    vessel_df = pd.DataFrame(vessel.logbook)
    waiting_stop = vessel_df[vessel_df.Message == "Waiting stop"]
    if not waiting_stop.empty:
        delay = (waiting_stop.Timestamp-vessel.metadata["arrival_time"]).iloc[0]
    else:
        delay = pd.Timedelta(seconds=0)
    delays.append(delay)

In [16]:
print(f"The average vessel delay is {np.round(np.average(delays).total_seconds()/60,1)} minutes")

The average vessel delay is 0.0 minutes
